# Initialization

Every parameter in a network has a value before training starts: a fully
connected layer mapping 512 inputs to 256 outputs brings a $256 \times 512$
weight matrix filled with random numbers. that section explains why the scale of those
numbers decides whether a deep network trains at all, and derives the Xavier
[@Glorot.Bengio.2010] and He [@He.Zhang.Ren.ea.2015] schemes that
keep signal variance stable across layers. This section is the practical
companion: what the library does when you say nothing, how to impose a scheme
of your own on a whole model or a single block, what transformer-era code adds
on top of Xavier and He, and how to write an initializer that no menu provides.

In [1]:
import math
import tensorflow as tf

## Defaults and When to Override Them

You can usually ignore initialization because the default is sensible.

Keras attaches no numbers at construction either: `Dense(256)` fixes only the
output width, and the kernel comes into existence at build time, when the
layer first learns its input shape (every Keras layer is lazy in the sense of
that section). The build draws Glorot uniform, the Xavier scheme
of that section in its uniform variant: the kernel comes from
$U(-b, b)$ with $b = \sqrt{6/(n_\textrm{in} + n_\textrm{out})}$, and the bias
starts at zero. A uniform distribution on $[-b, b]$ has standard deviation
$b/\sqrt{3}$, so all three claims are cheap to check against a fresh layer:

In [2]:
tf.keras.utils.set_random_seed(0)
layer = tf.keras.layers.Dense(256)
layer.build((None, 512))
w = layer.kernel.numpy()
bound = math.sqrt(6 / (512 + 256))
print(f'range [{w.min():.4f}, {w.max():.4f}], predicted bound {bound:.4f}')
print(f'std {w.std():.4f}, predicted {bound / math.sqrt(3):.4f}')
print(f'bias all zero: {bool((layer.bias.numpy() == 0).all())}')

range [-0.0884, 0.0884], predicted bound 0.0884
std 0.0511, predicted 0.0510
bias all zero: True


Other libraries make different but equally fan-aware choices, with one legacy
exception:

| Library | Default for a dense layer's weight |
|:--|:--|
| PyTorch | $U(-1/\sqrt{n_\textrm{in}},\, 1/\sqrt{n_\textrm{in}})$, bias likewise |
| Flax (JAX) | LeCun normal: truncated normal, std $1/\sqrt{n_\textrm{in}}$; zero bias |
| Keras (TensorFlow) | Glorot uniform: $U(\pm\sqrt{6/(n_\textrm{in} + n_\textrm{out})})$; zero bias |
| MXNet Gluon | $U(-0.07, 0.07)$, not fan-aware, so override it; zero bias |

When should you override? Four situations recur: a deep network without
normalization layers, where variance compounds across depth (we demonstrate
this below); reproducing a paper whose results depend on its initialization
recipe; architecture-specific corrections such as the residual scaling later
in this section; and parameters you create by hand, where Keras quietly
substitutes a default of its own, since `add_weight` falls back to Glorot
uniform unless you pass an `initializer`, a sensible scale for a weight
matrix and the wrong one for almost anything else. There is no
construction-time fine print to remember: fan-in and fan-out are read off
the input shape when `build` runs.

## Applying Initializers

To impose a scheme we declare it where the layer is declared: every Keras
layer accepts `kernel_initializer` and `bias_initializer` arguments, each an
object mapping a shape and dtype to a tensor, and `tf.keras.initializers`
supplies the standard menu (the string `'zeros'` names the same object as
`tf.keras.initializers.Zeros()`). The arguments are stored at construction
and run at build time, so they wait for lazy shapes automatically. That
covers models you are about to build. The second route, for a model that
already exists, rests on the fact that a kernel is a mutable variable:
iterate over `model.layers`, decide by type what to do with each layer, and
overwrite the kernels you select in place with `assign`. We show both,
starting with constructor arguments.

In [3]:
init_normal = tf.keras.initializers.RandomNormal(stddev=0.01)
net = tf.keras.Sequential([
    tf.keras.layers.Dense(8, kernel_initializer=init_normal,
                          bias_initializer='zeros'),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Dense(1, kernel_initializer=init_normal,
                          bias_initializer='zeros')])

X = tf.ones((2, 4))
net(X)
net.layers[0].kernel[:, 0], net.layers[0].bias[0]

(<tf.Tensor: shape=(4,), dtype=float32, numpy=array([ 0.01419753,  0.01572837, -0.02287325, -0.00140878], dtype=float32)>,
 <tf.Tensor: shape=(), dtype=float32, numpy=0.0>)

Mixing schemes per block needs no dispatch while the model is under
construction: each layer names its own initializer, so the choice sits
exactly where the layer is declared. Below we give the first layer Xavier
initialization (that section), which Keras spells `GlorotUniform`
after its author, and set the last to a constant, a poor idea for training by
the symmetry argument of that section, but it makes the
mechanics visible:

In [4]:
net = tf.keras.Sequential([
    tf.keras.layers.Dense(
        8, kernel_initializer=tf.keras.initializers.GlorotUniform()),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Dense(
        1, kernel_initializer=tf.keras.initializers.Constant(42.0))])

net(X)
net.layers[0].kernel[:, 0], net.layers[2].kernel[:, 0]

(<tf.Tensor: shape=(4,), dtype=float32, numpy=array([ 0.22517478, -0.18920076,  0.62674755,  0.11356837], dtype=float32)>,
 <tf.Tensor: shape=(8,), dtype=float32, numpy=array([42., 42., 42., 42., 42., 42., 42., 42.], dtype=float32)>)

The second route re-initializes a model that already exists. The loop below
is the whole mechanism: iterate `model.layers`, dispatch with `isinstance` so
that activations and anything else without a kernel are skipped, and `assign`
a fresh draw into each kernel. `assign` overwrites the variable's buffer in
place, which is what we want, since the parameter must remain the same
variable the model tracks and any existing optimizer updates. The function
repairs the constant-42 network from the previous cell; biases pass through
untouched:

In [5]:
def reinit(model, init):
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dense):
            layer.kernel.assign(init(layer.kernel.shape))

reinit(net, tf.keras.initializers.GlorotUniform(seed=1))
net.layers[2].kernel[:3]

<tf.Tensor: shape=(3, 1), dtype=float32, numpy=
array([[ 0.6180687 ],
       [-0.06535196],
       [ 0.16377491]], dtype=float32)>

## Modern Schemes: Truncation, Depth, and Zeros

Xavier and He set the variance of a single layer. The schemes below, standard
in transformer codebases, adjust what happens in the distribution's tails,
across depth, and at a block's start.

### Truncated Normals

A Gaussian gets the variance right, but its tails are unbounded. That is
harmless for one draw and a near-certainty at scale: among the $10^8$ weights
of a BERT-sized model, dozens land beyond five standard deviations. Large
draws also consume disproportionate dynamic range once a model is cast to low
precision (that section). BERT and implementations in the ViT
lineage use truncated-normal initialization
[@Devlin.Chang.Lee.ea.2018; @Dosovitskiy.Beyer.Kolesnikov.ea.2021]: the
tails are cut off at a fixed multiple of the nominal standard deviation. Raw
truncated-normal initializers do not necessarily restore the variance removed
with the tails; fan-aware variance-scaling initializers often do.

`tf.keras.initializers.TruncatedNormal` fixes its cutoff at two standard
deviations: draws landing outside the bound are discarded and redrawn, and no
rescaling follows. Truncation also backs the fan-aware factory behind
`HeNormal` and its relatives, which draws a truncated normal and then
rescales it so the standard deviation lands on target despite the missing
tails:

In [6]:
init = tf.keras.initializers.RandomNormal(stddev=0.02, seed=0)
w = init((1000, 1000)).numpy()
print(f'normal:    std {w.std():.4f}, max weight {abs(w).max():.4f}')
init = tf.keras.initializers.TruncatedNormal(stddev=0.02, seed=0)
w = init((1000, 1000)).numpy()
print(f'truncated: std {w.std():.4f}, max weight {abs(w).max():.4f}')

normal:    std 0.0200, max weight 0.1046
truncated: std 0.0176, max weight 0.0400


A million plain-normal draws produce entries near $5\sigma = 0.1$; the
truncated version guarantees $|w| \leq 0.04$. Its printed standard deviation
dips slightly below the nominal 0.02 because truncation removes tail mass;
practice ignores the difference.

### Scaling Down Residual Branches

A residual network computes $\mathbf{x}_{k+1} = \mathbf{x}_k +
f_k(\mathbf{x}_k)$, so its output is the input plus the sum of $N$ block
contributions. If every block is initialized identically, each contribution
has variance proportional to that of the already-inflated stream it reads, and
the stream's variance compounds geometrically with depth. This is the
depth-wise cousin of the layer-wise explosion in
that section. GPT-2's fix is to shrink only the *last*
linear layer of each residual branch, the output projection that writes into
the stream, by a factor of $1/\sqrt{N}$: with $N$ roughly independent
contributions each scaled down that way, the sum's variance stays $O(1)$
regardless of depth [@Radford.Wu.Child.ea.2019]. For an $L$-layer
transformer the published recipe is a normal with std $0.02/\sqrt{2L}$ on the
residual projections — the number of contributions is $N = 2L$, because each
layer writes to the stream twice, once from attention and once from the MLP.
Unlike Xavier or He, the base 0.02 is not derived from fan-in; it is an
empirical constant that worked at GPT-2's widths and stuck.

the figure draws the two regimes side by side.

![A residual stream with additive block contributions, unscaled (left) versus scaled by 1/sqrt(N) (right). Under the independent-contribution approximation, unscaled variance grows with N and scaling keeps the sum O(1). In an unnormalized stack, each branch reads an already inflated stream, so growth can be faster, as the experiment below shows.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-residual-stream.svg)

### Starting a Block at Zero

Zero-initializing *every* weight is fatal: all units in a layer compute the
same output, receive the same gradient, and stay identical forever
(that section). Zero-initializing just the *last* layer
of a residual block is a different and useful move. The branch then
contributes exactly nothing, each block is the identity map, and the network
starts as a shallow function whose depth switches on gradually during
training. Symmetry is not a problem because the branch's earlier layers keep
their random weights. On the first backward pass, the zeroed projection
receives a nonzero gradient because its input is nonzero, while the earlier
layers receive zero gradient through that projection. After the projection
moves off zero, gradient reaches the whole branch. Keep this scheme distinct
from the previous one: GPT-2 makes every residual projection *small but
nonzero*, whereas zero-init makes one layer *exactly zero* so the block starts
as an exact identity.

### Watching the Variance Compound

Claims about variance at depth are cheap to test. We reuse a compact residual
block (it repeats the one from that section) and stack
it $N$ deep:

In [7]:
class ResidualBlock(tf.keras.Model):
    def __init__(self, d, out_init='he_normal'):
        super().__init__()
        self.body = tf.keras.Sequential([
            tf.keras.layers.Dense(4 * d, kernel_initializer='he_normal'),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Dense(d, kernel_initializer=out_init)])

    def call(self, X):
        return X + self.body(X)

Every dense layer gets He initialization, appropriate for the ReLU inside
the branch; the string `'he_normal'` names the same object as
`tf.keras.initializers.HeNormal()`. The treatment is a constructor argument
as well: the block takes its output projection's initializer as a parameter,
so each of the three treatments is just a different initializer. Leaving it
alone means He again and zeroing is `'zeros'`, but scaling by $1/\sqrt{N}$
has no menu entry, so we subclass `Initializer`: the instance stores the
depth, and its `__call__` shrinks a fresh He draw, behind the same
shape-and-dtype signature everything on the menu implements.

In [8]:
class ScaledHe(tf.keras.initializers.Initializer):
    def __init__(self, num_blocks):
        self.num_blocks = num_blocks

    def __call__(self, shape, dtype=None):
        w = tf.keras.initializers.HeNormal()(shape, dtype)
        return w * self.num_blocks ** -0.5

def output_std(num_blocks, out_init):
    tf.keras.utils.set_random_seed(0)
    net = tf.keras.Sequential(
        [ResidualBlock(64, out_init=out_init(num_blocks))
         for _ in range(num_blocks)])
    X = tf.random.normal((256, 64), seed=1)
    return float(tf.math.reduce_std(net(X)))

One forward pass on unit-variance inputs per depth and treatment:

In [9]:
tweaks = {'default': lambda n: 'he_normal',
          'scaled': ScaledHe,
          'zero': lambda n: 'zeros'}
print(f'{"N":>3}' + ''.join(f'{name:>10}' for name in tweaks))
for n in (2, 8, 32):
    stds = (output_std(n, tweak) for tweak in tweaks.values())
    print(f'{n:>3}' + ''.join(f'{s:>10.3g}' for s in stds))

  N   default    scaled      zero


  2      3.24      1.91         1


  8      99.5      2.18         1


 32  7.56e+07      2.86         1


The default column multiplies by a roughly constant factor per block in this
unnormalized stack, reaching tens of millions by $N=32$. Such initial
activations make optimization impractical. The scaled column stays near a
small constant over the depths tested: scaling cuts each branch's variance by
a factor of $N$, following the independent-contribution approximation above.
The zero column reproduces the input's standard deviation, since every block
is the identity. This forward-pass test does not prove how an optimizer will
behave, but it catches an unusable initialization before training begins.

## Custom Initializers

Occasionally the menu has nothing you need.

An initializer is just an object mapping `(shape, dtype)` to a tensor:
subclass `tf.keras.initializers.Initializer`, implement `__call__`, and every
layer will accept an instance, so writing one is no harder than using one.

Suppose, to make the point vividly, we want weights distributed as

$$
\begin{aligned}
    w \sim \begin{cases}
        U(5, 10) & \textrm{ with probability } \frac{1}{4} \\
            0    & \textrm{ with probability } \frac{1}{2} \\
        U(-10, -5) & \textrm{ with probability } \frac{1}{4}
    \end{cases}
\end{aligned}
$$

Draw uniformly from $U(-10, 10)$, then zero every entry of magnitude below 5.
Handing an instance to `kernel_initializer` makes it official:

In [10]:
class MyInit(tf.keras.initializers.Initializer):
    def __call__(self, shape, dtype=None):
        w = tf.random.uniform(shape, -10, 10, dtype=dtype)
        return w * tf.cast(tf.abs(w) >= 5, w.dtype)

net = tf.keras.Sequential([
    tf.keras.layers.Dense(8, kernel_initializer=MyInit()),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Dense(1)])

net(X)
net.layers[0].kernel[:2]

<tf.Tensor: shape=(2, 8), dtype=float32, numpy=
array([[-0.      , -5.868671,  0.      ,  0.      , -0.      ,  6.156559,
        -0.      ,  9.962585],
       [ 0.      , -7.492528,  0.      ,  0.      ,  0.      , -0.      ,
        -0.      ,  0.      ]], dtype=float32)>

No guard is needed on the way in, because `assign` sits outside automatic
differentiation: TensorFlow records gradients on a `GradientTape`, and an
assignment is a write to the variable's buffer, not an operation on the tape.
The same door handles one-off surgery, such as offsetting a whole matrix or
pinning a single entry, since slicing a variable composes with `assign`:

In [11]:
w = net.layers[0].kernel
w.assign(w + 1)
w[0, 0].assign(42)
w[0]

<tf.Tensor: shape=(8,), dtype=float32, numpy=
array([42.      , -4.868671,  1.      ,  1.      ,  1.      ,  7.156559,
        1.      , 10.962585], dtype=float32)>

The ordering caveat of the lazy world (that section) splits by
route here: a constructor initializer is stored and runs at build time, so it
can never fire too early, but `assign` surgery needs a kernel to exist and
must therefore follow the first call (or an explicit `build`). The cells
above respected that order by running `net(X)` before touching
`net.layers[0].kernel`.

## Summary

Keras initializes parameters at build time, when a layer first learns its
input shape, with fan-aware defaults; override them when depth or a paper's
recipe demands it. The mechanism is one pattern: an initializer is an object
mapping `(shape, dtype)` to a tensor, handed to a layer as
`kernel_initializer` at construction or, for a model that already exists,
written into each selected kernel by a `model.layers` walk and `assign`.

On top of Xavier and He,
transformer-era code truncates normal tails to bound outliers, shrinks each
residual output projection by $1/\sqrt{N}$ so the stream's variance stays
$O(1)$ at any depth, and zero-initializes a block's last layer to start it as
the identity.

Anything the menu lacks is an `Initializer` subclass with a few lines of
`tf.random` code in its `__call__`.

## Exercises

1. Instrument the residual stack: record the standard deviation of the
   activation after every block (run the stack one block at a time, or
   capture per-block activations with the tools of that section)
   for the default and scaled treatments at $N=32$, and plot it against
   depth. Which curve matches the geometric-growth prediction?
1. Zero-initialize *all* layers of every block instead of just the output
   projection. The forward pass still returns the input, but what can the
   network learn? Work out which parameters receive a nonzero gradient, and
   relate the answer to the symmetry-breaking argument of
   that section.
1. Write an initializer that fills each parameter from a dictionary keyed by
   parameter name (walk `net.named_parameters()` in PyTorch, `net.weights`
   in TensorFlow, `nnx.iter_modules(net)` in JAX, `net.collect_params()`
   in MXNet). You have re-invented part of checkpoint loading,
   which that section covers.
1. For a normal distribution truncated at $\pm 2\sigma$: what fraction of
   draws does the clip discard, and by how much does it shrink the standard
   deviation? Verify both numbers against the printed output of the truncation
   demo above.